In [ ]:
import aiohttp
import aiofiles
import asyncio
import json
import re
import pandas as pd

#Gọi lại ra list IDs
list_id=pd.read_csv("D:\products-0-200000.csv")
print(list_id)

# Thiết lập giới hạn số lần gọi API đồng thời
SEMAPHORE_LIMIT = 100
semaphore = asyncio.Semaphore(SEMAPHORE_LIMIT)


def standardize_description(html_content):
    if not html_content:
        return ""

    cleaned_content = re.sub(r'<script.*?>.*?</script>', '', html_content, flags=re.DOTALL)
    cleaned_content = re.sub(r'<style.*?>.*?</style>', '', cleaned_content, flags=re.DOTALL)
    cleaned_content = re.sub(r'</p>|<br/?>', '\n', cleaned_content)
    cleaned_content = re.sub(r'<[^>]+>', '', cleaned_content)

    from html import unescape
    cleaned_content = unescape(cleaned_content)

    cleaned_content = re.sub(r'\s+', ' ', cleaned_content).strip()

    return cleaned_content


async def fetch_product_data(session, product_id):
    url = f"https://api.tiki.vn/product-detail/api/v1/products/{product_id}"
    try:
        async with semaphore:
            async with session.get(url, timeout=30) as response:
                if response.status == 200:
                    data = await response.json()
                    product_info = {
                        "id": data.get("id"),
                        "name": data.get("name"),
                        "url_key": data.get("url_key"),
                        "price": data.get("price"),
                        "description": standardize_description(data.get("description")),
                        "images_url": [img["base_url"] for img in data.get("images", [])]
                    }
                    return product_info
                else:
                    error_message = f"Mã lỗi HTTP: {response.status}"
                    return (product_id, error_message)
    except (aiohttp.ClientError, asyncio.TimeoutError) as e:
        error_message = f"Lỗi kết nối: {e}"
        return (product_id, error_message)


async def main():
    batch_size = 1000
    total_products = len(list_id)

    # 1. Tải danh sách các ID đã xử lý từ file log (nếu có)
    processed_ids_path = "processed_ids.json"
    processed_ids = set()
    try:
        async with aiofiles.open(processed_ids_path, 'r', encoding='utf-8') as f:
            processed_ids = set(json.loads(await f.read()))
            print(f"Đã tải {len(processed_ids)} ID đã xử lý từ phiên trước.")
    except (FileNotFoundError, json.JSONDecodeError):
        print("Không tìm thấy file log ID đã xử lý. Bắt đầu từ đầu.")

    # Lọc ra các ID chưa được xử lý
    remaining_ids = [pid for pid in list_id if pid not in processed_ids]
    total_remaining = len(remaining_ids)

    failed_products_log = []

    async with aiohttp.ClientSession() as session:
        for i in range(0, total_remaining, batch_size):
            batch_ids = remaining_ids[i:i + batch_size]

            # 2. Xử lý các ID và cập nhật danh sách đã xử lý
            batch_tasks = [fetch_product_data(session, pid) for pid in batch_ids]
            results = await asyncio.gather(*batch_tasks)

            valid_products = []
            for result in results:
                if isinstance(result, tuple):
                    product_id, error_reason = result
                    failed_products_log.append({
                        "product_id": product_id,
                        "reason": error_reason
                    })
                    print(f"Lỗi: Sản phẩm {product_id}, Lý do: {error_reason}")
                else:
                    valid_products.append(result)
                    # Thêm ID đã xử lý thành công vào set
                    processed_ids.add(result["id"])

            # Lưu dữ liệu của batch và cập nhật file log ID
            if valid_products:
                file_number = (i // batch_size) + 1
                filename = f"products_{file_number:03d}.json"
                try:
                    async with aiofiles.open(filename, 'w', encoding='utf-8') as f:
                        await f.write(json.dumps(valid_products, ensure_ascii=False, indent=4))
                    print(f"Đã lưu thành công {len(valid_products)} sản phẩm vào file {filename}.")

                    # 3. Ghi lại các ID đã xử lý thành công
                    async with aiofiles.open(processed_ids_path, 'w', encoding='utf-8') as f:
                        await f.write(json.dumps(list(processed_ids), ensure_ascii=False, indent=4))

                except IOError as e:
                    print(f"Lỗi khi lưu file {filename} hoặc file log: {e}")

    if failed_products_log:
        try:
            async with aiofiles.open("failed_products.json", 'w', encoding='utf-8') as f:
                await f.write(json.dumps(failed_products_log, ensure_ascii=False, indent=4))
            print(f"\nĐã lưu {len(failed_products_log)} sản phẩm bị lỗi vào file failed_products.json.")
        except IOError as e:
            print(f"Lỗi khi lưu file lỗi: {e}")


if _name_ == "_main_":
    asyncio.run(main())